In [2]:
import os
import torch
from google.colab import drive
from transformers import pipeline


drive.mount('/content/drive')

zip_path_on_drive = '/content/drive/MyDrive/nlp/models/my_toxic_rubert.zip'
local_extract_path = '/content/my_toxic_rubert'


if os.path.exists(zip_path_on_drive):

    !unzip -q -o {zip_path_on_drive} -d /
    print("Archive found on Google drive and unpacked")
else:
    print(f"Error: Archive not found on path {zip_path_on_drive}. ")


if os.path.exists(local_extract_path):
    classifier = pipeline(
        "text-classification",
        model=local_extract_path,
        tokenizer=local_extract_path,
        device=0 if torch.cuda.is_available() else -1
    )
    print("Model downloaded to memory and ready")

Mounted at /content/drive
Archive found on Google drive and unpacked


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model downloaded to memory and ready


In [3]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification, pipeline


snlp_model_name = 's-nlp/russian_toxicity_classifier'


snlp_tokenizer = BertTokenizer.from_pretrained(snlp_model_name)
snlp_model = BertForSequenceClassification.from_pretrained(snlp_model_name)


snlp_classifier = pipeline(
    "text-classification",
    model=snlp_model,
    tokenizer=snlp_tokenizer,
    device=0 if torch.cuda.is_available() else -1
)
print("SNLP baseline model successfully loaded and ready for comparison!")


def test_snlp_model(text):
    result = snlp_classifier(text)[0]
    label = result['label']
    score = result['score']

    if label.lower() == 'toxic':
        return f"toxic (SNLP confidence: {score:.2%})"
    else:
        return f"normal (SNLP confidence: {score:.2%})"


print("\n--- Testing the model ---")
print(f"Text: 'Ты просто супер!' -> {test_snlp_model('Ты просто супер!')}")
print(f"Text: 'Ну ты и урод волосатый' -> {test_snlp_model('Ну ты и урод волосатый')}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.40M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.04k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/711M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: s-nlp/russian_toxicity_classifier
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SNLP baseline model successfully loaded and ready for comparison!

--- Testing the model ---
Text: 'Ты просто супер!' -> normal (SNLP confidence: 78.92%)
Text: 'Ну ты и урод волосатый' -> toxic (SNLP confidence: 99.28%)


In [6]:
import pandas as pd

# 1. Define the path to your balanced dataset
dataset_path = '/content/drive/MyDrive/nlp/balanced_toxic_dataset.csv'

print("Loading dataset from Google Drive...")
df_balanced = pd.read_csv(dataset_path)

# 2. Re-shuffle the data using the same random_state to keep the split consistent
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# 3. Restore the 60/20/20 split boundaries
total_len = len(df_balanced)
val_end = int(0.8 * total_len) # End of Train + Validation sets

# The test set consists of the last 20%
df_test = df_balanced.iloc[val_end:].copy()

# 4. Standardize column names (Hugging Face expects 'text' and 'label')
if 'comment' in df_test.columns and 'toxic' in df_test.columns:
    df_test = df_test.rename(columns={'comment': 'text', 'toxic': 'label'})

print(f"✅ Test dataset successfully restored!")
print(f"Total rows in file: {total_len}")
print(f"Test set size (df_test): {len(df_test)} rows.")

# Preview the first 3 rows
display(df_test.head(3))

Loading dataset from Google Drive...
✅ Test dataset successfully restored!
Total rows in file: 98862
Test set size (df_test): 19773 rows.


,text,label
79089,"ну пять рублей, ну 98 год. и?",0
79090,народу затуманили мозги...кругом были смотрящи...,1
79091,я что то не пойму за что поддерживать? мне воо...,1


In [8]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score



texts = df_test['text'].tolist()
true_labels = df_test['label'].tolist()

# 2. Get predictions from YOUR model
print("⏳ 1/2: Your model is making predictions...")

my_preds_raw = classifier(texts, batch_size=32, truncation=True, max_length=128)

# Your model outputs 'LABEL_1' (toxic) and 'LABEL_0' (normal)
my_preds = [1 if p['label'] == 'LABEL_1' else 0 for p in my_preds_raw]

# 3. Get predictions from the SNLP baseline model
print("⏳ 2/2: SNLP model is making predictions...")
snlp_preds_raw = snlp_classifier(texts, batch_size=32, truncation=True, max_length=128)

# The SNLP model outputs 'toxic' and 'neutral'
snlp_preds = [1 if p['label'].lower() == 'toxic' else 0 for p in snlp_preds_raw]


print("\nCALCULATING METRICS...\n")

def get_metrics(y_true, y_pred):
    return {
        "Accuracy": round(accuracy_score(y_true, y_pred), 4),
        "F1-Score": round(f1_score(y_true, y_pred), 4),
        "Precision": round(precision_score(y_true, y_pred), 4),
        "Recall": round(recall_score(y_true, y_pred), 4)
    }

my_metrics = get_metrics(true_labels, my_preds)
snlp_metrics = get_metrics(true_labels, snlp_preds)

# Create a comparison DataFrame
df_comparison = pd.DataFrame([my_metrics, snlp_metrics],
                             index=["🏆 My Model (Fine-Tuned)", "🤖 SNLP (Baseline)"])

# Display the table
display(df_comparison)


#  ERROR ANALYSIS (Where does your model perform better?)

df_analysis = pd.DataFrame({
    'text': texts,
    'true_label': true_labels,
    'my_pred': my_preds,
    'snlp_pred': snlp_preds
})

# Find cases where your model was correct, but SNLP was wrong
my_wins = df_analysis[(df_analysis['my_pred'] == df_analysis['true_label']) &
                      (df_analysis['snlp_pred'] != df_analysis['true_label'])]

print(f"\n Found {len(my_wins)} comments where YOUR model was correct and SNLP failed!")
print("Here are 10 examples of your model's success:")
display(my_wins[['text', 'true_label']].head(10))

⏳ 1/2: Your model is making predictions...
⏳ 2/2: SNLP model is making predictions...

CALCULATING METRICS...



,Accuracy,F1-Score,Precision,Recall
🏆 My Model (Fine-Tuned),0.9844,0.9846,0.9831,0.9861
🤖 SNLP (Baseline),0.9686,0.9682,0.9881,0.9491



 Found 456 comments where YOUR model was correct and SNLP failed!
Here are 10 examples of your model's success:


,text,true_label
30,У него явно вышел новый альбом...\n,1
61,"назим, твоей маме хочется сказать што она плох...",1
156,надо мальчика кастрировать🙂,1
182,михаил меченый пособник дьявала.сколько пакост...,1
279,левые радикалы нацисты\n,1
305,это чума если есть рядом другие свини и не пре...,1
348,"он просто алкаш, и наркоман . клоун. должен от...",1
350,растерять этих западных подкоблучьников,1
382,чудовище всё смердишь.,1
465,откуда эта страшная обезьяна там али - баба вз...,1
